# 🔧 Output Parsers

## What is an Output Parser?
LLMs return text. You often need **structured data**.

Output parsers transform LLM text output into:
- Plain strings
- JSON/dicts
- Pydantic models
- Lists
- Booleans
- Enums

## Why Parsers?
Raw LLM output: `"The answer is 42, because the universe..."`
Your app needs: `{"answer": 42}`

Parsers bridge this gap.

## Parser Types
| Parser | Output | Use Case |
|--------|--------|----------|
| StrOutputParser | str | Simple text |
| JsonOutputParser | dict | Structured data |
| PydanticOutputParser | Pydantic model | Typed, validated |
| CommaSeparatedListOutputParser | list[str] | Simple lists |
| BooleanOutputParser | bool | Yes/No questions |
| DatetimOutputParser | datetime | Date extraction |


In [ ]:
# ── StrOutputParser ─────────────────────────────────────────────────────
# WHY: LLM returns AIMessage. StrOutputParser extracts .content → str.
# This is the most common parser — use it unless you need structure.

from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage

parser = StrOutputParser()

# Simulate LLM output
ai_msg = AIMessage(content='Python is a high-level programming language.')

# Parse
result = parser.invoke(ai_msg)
print('Input type:', type(ai_msg).__name__)  # AIMessage
print('Output type:', type(result).__name__) # str
print('Output:', result)

In [ ]:
# ── PydanticOutputParser ─────────────────────────────────────────────────
# WHY: Use when you need TYPED, VALIDATED structured output.
# LangChain will inject format instructions into your prompt automatically!

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

# Define your output schema
class PersonInfo(BaseModel):
    name: str = Field(description='Full name of the person')
    age: int = Field(description='Age in years')
    occupation: str = Field(description='Job or profession')

# Create parser
parser = PydanticOutputParser(pydantic_object=PersonInfo)

# Parser can generate format instructions!
print('Format instructions (auto-injected into prompt):')
print(parser.get_format_instructions()[:300], '...')

# Build chain
prompt = ChatPromptTemplate.from_messages([
    ('system', 'Extract person info from text.\n{format_instructions}'),
    ('human', '{text}'),
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | ChatOpenAI(model='gpt-4o-mini', temperature=0) | parser

result = chain.invoke({'text': 'Alice Smith is a 28-year-old software engineer.'})
print('\nParsed result:', result)
print('Type:', type(result).__name__)  # PersonInfo
print('Name:', result.name)
print('Age:', result.age)

In [ ]:
# ── JsonOutputParser ─────────────────────────────────────────────────────
# WHY: When you want dicts but don't need Pydantic validation.
# More flexible than PydanticOutputParser.

from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

parser = JsonOutputParser()

chain = (
    ChatPromptTemplate.from_template(
        'Return a JSON object with keys: capital, population, language for {country}. '
        'Return ONLY valid JSON, no other text.'
    )
    | ChatOpenAI(model='gpt-4o-mini', temperature=0)
    | parser
)

result = chain.invoke({'country': 'Japan'})
print('Type:', type(result).__name__)  # dict!
print('Result:', result)
print('Capital:', result.get('capital'))

In [ ]:
# ── Structured Output (Modern Approach) ─────────────────────────────────
# WHY: .with_structured_output() is the RECOMMENDED modern approach.
# It uses function calling / tool calling under the hood.
# More reliable than asking LLM to format JSON in its text response.

from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

class Recipe(BaseModel):
    name: str = Field(description='Recipe name')
    ingredients: list[str] = Field(description='List of ingredients')
    steps: list[str] = Field(description='Cooking steps')
    prep_time_minutes: int = Field(description='Prep time in minutes')

# Modern approach — no manual parser needed!
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
structured_llm = llm.with_structured_output(Recipe)  # returns Recipe directly

result = structured_llm.invoke('Give me a simple pasta recipe')
print('Type:', type(result).__name__)  # Recipe
print('Name:', result.name)
print('Ingredients:', result.ingredients[:3], '...')
print('Prep time:', result.prep_time_minutes, 'minutes')

## 🎯 Interview Questions

1. **What is StrOutputParser and when would you use it?**
2. **Difference between PydanticOutputParser and with_structured_output()?**
3. **Why is with_structured_output() more reliable than JSON parsing?**
4. **What are format instructions in PydanticOutputParser?**
5. **What happens if an output parser fails?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Output Parsers — Structuring LLM Responses**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
